# 01. Mixed training: part of the test places joins training

Half of the external test parcels of every place (Madagascar, Senegal, Réunion, Antilles, France, Denmark, California) joins the European development parcels; the other half stays held out. Whole 0.1° blocks (~10 km) go to one side, per place and crop, so neighbouring fields never sit on both sides. Three split seeds are used.

Models, all scored on the same held-out half with the improved pipeline's selected features (`s2_harmonic_all_radar`):

| Model | Trained on |
|---|---|
| `europe_only` | European development parcels (as the improved pipeline) |
| `europe_plus_added` | European parcels + the added half of every place |
| `added_only` | the added half alone |
| `new_place` | European parcels + the added half of the *other* places; scored on each place in turn |

The improved pipeline's own test results stay its final scores. **Input:** `pipeline_outputs/improved/current_inputs.json`. **Output:** `pipeline_outputs/mixed_training/`.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.improved import mixing
pd.set_option("display.width", 200)

## Run the split experiment

In [ ]:
inputs_handoff = ""  # Optional improved inputs receipt; blank uses the current pointer
split_seeds = (1, 2, 3)
receipt = mixing.run_experiment(root, split_seeds, inputs_handoff or None, progress=lambda m: print(m, flush=True))
run = receipt.parent
print("Experiment receipt:", receipt)

## Parcels per place on each side

In [ ]:
split = pd.read_csv(run / "split.csv")
display(split.pivot_table(index="place", columns=["split_seed", "side"], values="parcels", fill_value=0))

## Scores on the held-out half (mean and range over split seeds)

In [ ]:
summary = pd.read_csv(run / "summary.csv")
metrics = ["macro_f1_all", "macro_f1_tropical", "macro_f1_without_france", "carrot_f1", "cabbage_f1", "potato_f1"]
order = ["europe_only", "europe_plus_added", "added_only", "new_place"]
mean = summary.groupby("model")[metrics].mean().reindex(order)
spread = summary.groupby("model")[metrics].agg(lambda v: v.max() - v.min()).reindex(order)
display(mean.round(3))
display(spread.round(3).add_suffix("_range"))
display(summary.groupby("model")["parcels"].mean().reindex(order).to_frame("held_out_parcels"))

## Recall per place and crop (mean over split seeds)

In [ ]:
recall = pd.read_csv(run / "recall_by_place.csv")
table = recall.groupby(["place", "crop", "model"]).agg(recall=("recall", "mean"), parcels=("parcels", "mean")).reset_index()
display(table.pivot_table(index=["place", "crop"], columns="model", values="recall").reindex(columns=order).round(2)
        .join(table.groupby(["place", "crop"])["parcels"].mean().rename("held_out_parcels")))

## Figure

In [ ]:
figure, axis = plt.subplots(figsize=(9, 4))
labels = ["All held-out", "Tropical", "Without France"]
width = .2
for k, model in enumerate(order):
    values = mean.loc[model, ["macro_f1_all", "macro_f1_tropical", "macro_f1_without_france"]].values
    axis.bar(np.arange(3) + (k - 1.5) * width, values, width, label=model)
axis.axhline(.5, color="#555", lw=.8, ls="--")
axis.set_xticks(range(3), labels)
axis.set_ylabel("Macro-F1 (held-out half)")
axis.set_ylim(0, 1)
axis.legend(frameon=False, fontsize=8)
figure.tight_layout()
figure.savefig(run / "macro_f1_by_model.png", dpi=150)
display(figure)
plt.close(figure)

## Atok maps

Two models use every external parcel: European + all test places, and the test places only. No independent test remains for these models; the maps are exploratory.

In [ ]:
maps_receipt = mixing.paoay_maps(root, inputs_handoff or None, progress=lambda m: print(m, flush=True))
maps = maps_receipt.parent
improved = json.loads((root / "pipeline_outputs/improved/current_paoay.json").read_text())["path"]
shares = {"europe_only (improved pipeline)": json.loads((Path(improved).parent / "paoay_summary.json").read_text())}
for name in mixing.MAP_MODELS:
    shares[name] = json.loads((maps / name / "paoay_summary.json").read_text())
table = pd.DataFrame({name: s["crop_shares_of_confident_cropland"] for name, s in shares.items()})
table["Benguet 2024"] = pd.Series(next(iter(shares.values()))["benguet_2024_shares"])
display(table.round(3))
display(pd.Series({name: s["confident_share_in_cropland"] for name, s in shares.items()}).round(3).to_frame("confident share of cropland"))
for name in mixing.MAP_MODELS:
    print(name)
    display(pd.read_csv(maps / name / "area_summary.csv"))

## Map preview

In [ ]:
import rasterio
from matplotlib.colors import ListedColormap
colors = ListedColormap(["#fcfcfb", "#eb6834", "#1baf7a", "#eda100", "#c3c2b7", "#e4e3df"])
folders = {"europe_only": Path(improved).parent, **{name: maps / name for name in mixing.MAP_MODELS}}
figure, axes = plt.subplots(1, len(folders), figsize=(5 * len(folders), 5))
for axis, (name, folder) in zip(axes, folders.items()):
    with rasterio.open(folder / "crop_map.tif") as raster:
        axis.imshow(raster.read(1), cmap=colors, vmin=0, vmax=5, interpolation="nearest")
    axis.set_title(name, fontsize=9)
    axis.axis("off")
figure.suptitle("1 carrot (orange), 2 cabbage (green), 3 potato (yellow), 4 Unknown (grey)", fontsize=9)
figure.tight_layout()
figure.savefig(maps / "maps_preview.png", dpi=150)
display(figure)
plt.close(figure)